# 📧 Email & SMS Outreach Dashboard — L&D Designs
Works with the spreadsheet from **find_leads_multi.ipynb** or any previous leads file.
Uses **Email + SMS** as primary contact (no WhatsApp needed).

1. Run Cell 1 — upload your `multi_leads_*.xlsx` file
2. Run Cell 2 — builds the dashboard
3. Run Cell 3 — downloads it
4. Open the HTML in your browser and start outreaching

In [ ]:
# ── Cell 1: Upload spreadsheet ─────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
from google.colab import files
import openpyxl

print('Select your leads spreadsheet (.xlsx)...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(filename)
ws = wb.active
headers = [str(cell.value or '').strip() for cell in ws[1]]
print('Columns found:', headers)

all_leads = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[0]:
        all_leads.append(dict(zip(headers, row)))

print('Loaded', len(all_leads), 'leads.')

In [ ]:
# ── Cell 2: Build dashboard ─────────────────────────────────────────────
import re
from urllib.parse import quote
from datetime import datetime

# ── YOUR MESSAGES ──────────────────────────────────────────────────────
EMAIL_SUBJECT = 'Quick question — website for {name}'

EMAIL_BODY = '''Hi,

I noticed {name} doesn't have a website yet.

I build professional websites for local businesses in Wigan from just £199 — usually done within a week. No monthly fees, one-off payment.

Here's an example of my work: https://mellow-speculoos-d1850c.netlify.app/showcase.html

Would you be interested in a free quote?

Dylan
L&D Designs'''

SMS_MESSAGE = "Hi, I noticed {name} doesn't have a website. I build professional sites for Wigan businesses from £199 — done in days, no monthly fees. Free quote? - Dylan, L&D Designs"
# ───────────────────────────────────────────────────────────────────────

def get_field(lead, *keys):
    for k in keys:
        v = lead.get(k)
        if v and str(v).strip() and str(v).strip().lower() not in ('none','nan','null'):
            return str(v).strip()
    return ''

def clean_phone(phone):
    d = re.sub(r'[^\d+]', '', phone)
    if d.startswith('0'):
        d = '+44' + d[1:]
    elif d and not d.startswith('+'):
        d = '+44' + d
    return d

def has_contact(lead):
    return bool(get_field(lead, 'Email', 'email', 'Email Address') or
                get_field(lead, 'Phone', 'phone', 'Phone Number'))

# Filter: no website + has at least one contact method
leads = []
for l in all_leads:
    status = str(get_field(l, 'Website Status', 'Status', 'website_status') or '').upper()
    if 'ACTIVE' not in status and has_contact(l):
        leads.append(l)

print('Kept', len(leads), 'leads (no website + contactable)')
print('Skipped', len(all_leads) - len(leads), 'leads')

# Collect unique business types for filter buttons
all_types = sorted(set(
    get_field(l, 'Business Type', 'business_type')
    for l in leads
    if get_field(l, 'Business Type', 'business_type')
))

def make_card(lead, idx):
    name     = get_field(lead, 'Business Name', 'name') or 'Unknown'
    phone    = get_field(lead, 'Phone', 'phone', 'Phone Number')
    email    = get_field(lead, 'Email', 'email', 'Email Address')
    address  = get_field(lead, 'Address', 'address')
    biz_type = get_field(lead, 'Business Type', 'business_type')
    dist     = get_field(lead, 'Distance (mi)', 'Distance (miles)', 'distance_mi')
    card_id  = 'c' + str(idx)

    phone_clean = clean_phone(phone) if phone else ''
    maps_q = quote(name + ' ' + address)
    maps_link = 'https://www.google.com/maps/search/' + maps_q
    fb_link = 'https://www.google.com/search?q=' + quote(name + ' ' + address + ' facebook')

    # Email button
    if email and '@' in email:
        subj = quote(EMAIL_SUBJECT.format(name=name))
        body = quote(EMAIL_BODY.format(name=name))
        email_btn = '<a class="btn-email" href="mailto:' + email + '?subject=' + subj + '&body=' + body + '" onclick="setSent(\'' + card_id + '\')">&#9993; Email</a>'
    else:
        email_btn = '<span class="btn-noemail">No email</span>'

    # SMS button
    if phone_clean:
        sms_body = quote(SMS_MESSAGE.format(name=name))
        sms_link = 'sms:' + phone_clean + '&body=' + sms_body
        sms_btn = '<a class="btn-sms" href="' + sms_link + '" onclick="setSent(\'' + card_id + '\')">&#128172; SMS</a>'
    else:
        sms_btn = ''

    call_btn = ('<a class="btn-call" href="tel:' + phone + '">&#128222; Call</a>') if phone else ''
    maps_btn = '<a class="btn-maps" href="' + maps_link + '" target="_blank">&#128205; Maps</a>'
    fb_btn   = '<a class="btn-fb" href="' + fb_link + '" target="_blank">&#128269; Find</a>'

    type_badge = ('<span class="type-badge">' + biz_type + '</span>') if biz_type else ''
    dist_txt = (dist + ' mi') if dist else ''
    email_det = ('<div class="det">&#9993; ' + email + '</div>') if email else ''
    data_type = biz_type.lower().replace(' ','_').replace('&','').replace('/','_')

    return (
        '<div class="card" id="' + card_id + '" data-type="' + data_type + '">'
        '<div class="card-top">'
        '<div><div class="bname">' + name + '</div>'
        + type_badge +
        '<div class="addr">' + address + '</div></div>'
        '<div class="dist">' + dist_txt + '</div>'
        '</div>'
        + ('<div class="det">&#128222; ' + phone + '</div>' if phone else '') +
        email_det +
        '<div class="status-row">'
        '<select class="status-sel" onchange="saveStatus(\'' + card_id + '\')">'
        '<option value="new">&#128310; Not contacted</option>'
        '<option value="emailed">&#9993; Emailed</option>'
        '<option value="smsed">&#128172; SMS\'d</option>'
        '<option value="replied">&#128488; Replied</option>'
        '<option value="interested">&#9989; Interested</option>'
        '<option value="booked">&#127881; Booked!</option>'
        '<option value="no">&#10060; Not interested</option>'
        '</select>'
        '</div>'
        '<textarea class="notes" placeholder="Notes..." oninput="saveNotes(\'' + card_id + '\')" rows="2"></textarea>'
        '<div class="btns">'
        + email_btn + sms_btn + call_btn + maps_btn + fb_btn +
        '</div>'
        '</div>'
    )

cards_html = '\n'.join(make_card(l, i) for i, l in enumerate(leads))
total = len(leads)

# Type filter buttons
type_btns = '<button class="fbtn on" onclick="filtType(this,\'all\')" data-type="all">All</button>\n'
for t in all_types:
    t_key = t.lower().replace(' ','_').replace('&','').replace('/','_')
    count = sum(1 for l in leads if get_field(l,'Business Type','business_type') == t)
    type_btns += '<button class="fbtn" onclick="filtType(this,\'' + t_key + '\')" data-type="' + t_key + '">' + t + ' (' + str(count) + ')</button>\n'

# ── HTML ───────────────────────────────────────────────────────────────
html = '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
html += '<meta name="viewport" content="width=device-width,initial-scale=1">'
html += '<title>L&D Designs Outreach</title><style>'
html += '''
* { box-sizing:border-box; margin:0; padding:0; }
body { font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif; background:#f0f2f5; }
.header { background:#1a2035; color:white; padding:14px 20px; }
.header h1 { font-size:1.1rem; }
.header p  { opacity:.6; font-size:.75rem; margin-top:2px; }
.stats { display:flex; gap:8px; flex-wrap:wrap; padding:10px 16px;
          background:white; border-bottom:1px solid #e8e8e8; }
.stat { background:#f7f7f7; border-radius:6px; padding:7px 12px; text-align:center; }
.stat .n { font-size:1.3rem; font-weight:700; color:#1a2035; line-height:1; }
.stat .l { font-size:.65rem; color:#aaa; margin-top:2px; }
.topbar { display:flex; gap:6px; flex-wrap:wrap; align-items:center;
           padding:10px 16px; background:white; border-bottom:1px solid #e0e0e0; }
.topbar-label { font-size:.7rem; color:#999; letter-spacing:.05em; text-transform:uppercase; margin-right:4px; }
.fbtn { padding:5px 11px; border:2px solid #ddd; border-radius:14px;
         background:white; cursor:pointer; font-size:.73rem; font-weight:500; }
.fbtn.on { border-color:#1a2035; background:#1a2035; color:white; }
.divider { width:1px; height:20px; background:#e0e0e0; margin:0 4px; }
.grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(300px,1fr));
         gap:10px; padding:12px 16px; }
.card { background:white; border-radius:8px; padding:13px;
         box-shadow:0 1px 3px rgba(0,0,0,.06); border-left:3px solid #e74c3c; transition:opacity .3s; }
.card.status-emailed  { border-color:#3498db; }
.card.status-smsed    { border-color:#9b59b6; }
.card.status-replied  { border-color:#1abc9c; }
.card.status-interested{ border-color:#27ae60; }
.card.status-booked   { border-color:#f39c12; opacity:.7; }
.card.status-no       { opacity:.25; }
.card-top { display:flex; justify-content:space-between; gap:8px; margin-bottom:7px; }
.bname { font-weight:700; font-size:.9rem; color:#1a2035; }
.type-badge { display:inline-block; font-size:.62rem; background:#eef0f7; color:#555;
               padding:2px 7px; border-radius:10px; margin:3px 0 2px; font-weight:500; }
.addr  { font-size:.72rem; color:#aaa; margin-top:2px; }
.dist  { font-size:.72rem; color:#bbb; white-space:nowrap; flex-shrink:0; }
.det   { font-size:.76rem; color:#666; margin:2px 0; }
.status-row { margin:9px 0 5px; }
.status-sel { width:100%; padding:5px 7px; border:1px solid #e0e0e0;
               border-radius:5px; font-size:.78rem; background:#fafafa; cursor:pointer; }
.notes { width:100%; border:1px solid #e8e8e8; border-radius:5px;
          padding:5px 7px; font-size:.76rem; color:#555; resize:vertical;
          font-family:inherit; background:#fafafa; margin-bottom:7px; }
.notes:focus { outline:none; border-color:#1a2035; }
.btns { display:flex; gap:5px; flex-wrap:wrap; }
.btns a, .btns span { padding:6px 11px; border-radius:5px; text-decoration:none;
                       font-size:.74rem; font-weight:600; display:inline-flex; align-items:center; gap:4px; }
.btn-email  { background:#1a2035; color:white; }
.btn-email:hover { background:#2c3e6b; }
.btn-sms    { background:#9b59b6; color:white; }
.btn-sms:hover { background:#7d3c98; }
.btn-call   { background:#27ae60; color:white; }
.btn-call:hover { background:#1e8449; }
.btn-maps   { background:#f0f2f5; color:#555; border:1px solid #ddd; }
.btn-fb     { background:#f0f2f5; color:#555; border:1px solid #ddd; }
.btn-noemail{ background:#f0f0f0; color:#aaa; border:1px solid #e0e0e0; cursor:default; }
'''
html += '</style></head><body>'

html += '<div class="header"><h1>&#128247; L&amp;D Designs &mdash; Email &amp; SMS Outreach</h1>'
html += '<p>Businesses with no website &nbsp;&middot;&nbsp; ' + datetime.now().strftime('%d/%m/%Y') + ' &nbsp;&middot;&nbsp; Progress saves automatically</p></div>'

html += '<div class="stats">'
html += '<div class="stat"><div class="n">' + str(total) + '</div><div class="l">Total Leads</div></div>'
html += '<div class="stat"><div class="n" id="s-new">' + str(total) + '</div><div class="l">To Contact</div></div>'
html += '<div class="stat"><div class="n" id="s-contacted">0</div><div class="l">Contacted</div></div>'
html += '<div class="stat"><div class="n" id="s-interested">0</div><div class="l">Interested</div></div>'
html += '<div class="stat"><div class="n" id="s-booked">0</div><div class="l">Booked</div></div>'
html += '</div>'

# Status filter bar
html += '<div class="topbar">'
html += '<span class="topbar-label">Status:</span>'
html += '<button class="fbtn on" onclick="filtStatus(this,\'all\')">All</button>'
html += '<button class="fbtn" onclick="filtStatus(this,\'new\')">Not contacted</button>'
html += '<button class="fbtn" onclick="filtStatus(this,\'emailed\')">Emailed</button>'
html += '<button class="fbtn" onclick="filtStatus(this,\'smsed\')">SMS&\'d</button>'
html += '<button class="fbtn" onclick="filtStatus(this,\'replied\')">Replied</button>'
html += '<button class="fbtn" onclick="filtStatus(this,\'interested\')">Interested</button>'
html += '<button class="fbtn" onclick="filtStatus(this,\'booked\')">Booked</button>'
html += '</div>'

# Type filter bar
if all_types:
    html += '<div class="topbar" style="background:#fafafa;border-top:none">'
    html += '<span class="topbar-label">Type:</span>'
    html += type_btns
    html += '</div>'

html += '<div class="grid" id="grid">' + cards_html + '</div>'

html += '''
<script>
var STORE_KEY = 'ld_multi_outreach_v1';
var activeStatus = 'all';
var activeType   = 'all';

function getData() { try { return JSON.parse(localStorage.getItem(STORE_KEY)) || {}; } catch(e) { return {}; } }
function saveData(d) { localStorage.setItem(STORE_KEY, JSON.stringify(d)); }

function saveStatus(id) {
  var sel = document.querySelector('#'+id+' .status-sel');
  var val = sel.value;
  var d = getData(); if (!d[id]) d[id] = {};
  d[id].status = val; saveData(d);
  var card = document.getElementById(id);
  card.className = card.className.replace(/\bstatus-\S+/g,'').trim() + ' status-' + val;
  applyFilters(); updateStats();
}

function saveNotes(id) {
  var ta = document.querySelector('#'+id+' .notes');
  var d = getData(); if (!d[id]) d[id] = {};
  d[id].notes = ta.value; saveData(d);
}

function setSent(id) {
  setTimeout(function() {
    var sel = document.querySelector('#'+id+' .status-sel');
    if (sel && sel.value === 'new') {
      sel.value = 'emailed';
      saveStatus(id);
    }
  }, 1500);
}

function updateStats() {
  var counts = {new:0,emailed:0,smsed:0,replied:0,interested:0,booked:0,no:0};
  document.querySelectorAll('.status-sel').forEach(function(s) {
    if (counts[s.value] !== undefined) counts[s.value]++;
  });
  var el;
  el = document.getElementById('s-new');       if(el) el.textContent = counts.new;
  el = document.getElementById('s-contacted'); if(el) el.textContent = counts.emailed + counts.smsed + counts.replied;
  el = document.getElementById('s-interested');if(el) el.textContent = counts.interested;
  el = document.getElementById('s-booked');    if(el) el.textContent = counts.booked;
}

function filtStatus(btn, type) {
  document.querySelectorAll('.topbar:first-of-type .fbtn').forEach(function(b){ b.classList.remove('on'); });
  btn.classList.add('on');
  activeStatus = type;
  applyFilters();
}

function filtType(btn, type) {
  btn.parentElement.querySelectorAll('.fbtn').forEach(function(b){ b.classList.remove('on'); });
  btn.classList.add('on');
  activeType = type;
  applyFilters();
}

function applyFilters() {
  document.querySelectorAll('.card').forEach(function(card) {
    var sel  = card.querySelector('.status-sel');
    var stat = sel ? sel.value : 'new';
    var type = card.getAttribute('data-type') || '';
    var showStat = (activeStatus === 'all' || activeStatus === stat);
    var showType = (activeType === 'all' || activeType === type);
    card.style.display = (showStat && showType) ? '' : 'none';
  });
}

window.addEventListener('load', function() {
  var d = getData();
  Object.keys(d).forEach(function(id) {
    var card = document.getElementById(id);
    if (!card) return;
    if (d[id].status) {
      var sel = card.querySelector('.status-sel');
      if (sel) {
        sel.value = d[id].status;
        card.className = card.className.replace(/\bstatus-\S+/g,'').trim() + ' status-' + d[id].status;
      }
    }
    if (d[id].notes) {
      var ta = card.querySelector('.notes');
      if (ta) ta.value = d[id].notes;
    }
  });
  updateStats();
});
</script></body></html>'''

with open('outreach_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('Done!', total, 'leads in your dashboard.')
print('With email:', sum(1 for l in leads if get_field(l,'Email','email','Email Address')))
print('With phone/SMS:', sum(1 for l in leads if get_field(l,'Phone','phone','Phone Number')))

In [ ]:
# ── Cell 3: Download ──────────────────────────────────────────────────
from google.colab import files
files.download('outreach_dashboard.html')
print('Check your Downloads folder for outreach_dashboard.html')